
### AI Acknowledgement

Generative AI tools, including ChatGPT, were used as supporting tools during the development of this project. They assisted with selected coding tasks, debugging, explanations of machine learning concepts, and documentation.

# Hybrid Ensemble Learning Experiment

## TF-IDF + Metadata-Based Sentiment Classification

**Project:** EnsembleLab  
**Dataset:** Sri Lankan Tourism Review Sentiment-Rating Incongruence Dataset

### Objective
To investigate whether combining a text-based Logistic Regression classifier with a metadata-based Random Forest classifier can improve sentiment classification performance.

In [1]:
import pandas as pd
import numpy as np
import os

# Load the original dataset
dataset_path = "../datasets/Processed_Reviews_with_Sentiment.csv"

df = pd.read_csv(dataset_path)

print("DATASET LOADED SUCCESSFULLY")
print("=" * 50)
print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())
print("\nSentiment distribution:")
print(df["Sentiment"].value_counts())

DATASET LOADED SUCCESSFULLY
Dataset shape: (16156, 21)

Available columns:
['Location_Name', 'Located_City', 'Province', 'District', 'Location_Type', 'User_Locale', 'User_Country', 'Travel_Date_Month', 'Travel_Date_Year', 'Published_Date_Month', 'Published_Date_Year', 'User_Contributions', 'Rating', 'Helpful_Votes', 'Title', 'Text', 'Review_Length', 'Title_Length', 'Rating_Class', 'Review_Delay_Days', 'Sentiment']

Sentiment distribution:
Sentiment
POSITIVE    13041
NEGATIVE     1602
NEUTRAL      1513
Name: count, dtype: int64


In [2]:
# Remove exact duplicate records
df = df.drop_duplicates().reset_index(drop=True)

print("DATA CLEANING COMPLETED")
print("=" * 50)
print("Records after duplicate removal:", len(df))

print("\nSentiment distribution:")
print(df["Sentiment"].value_counts())

print("\nMissing values:", df.isnull().sum().sum())

DATA CLEANING COMPLETED
Records after duplicate removal: 16155

Sentiment distribution:
Sentiment
POSITIVE    13040
NEGATIVE     1602
NEUTRAL      1513
Name: count, dtype: int64

Missing values: 0


In [3]:
from sklearn.preprocessing import LabelEncoder

# Use exactly the same features as the original experiment
selected_features = [
    "Text",
    "Title",
    "Location_Type",
    "Province",
    "Location_Name",
    "User_Country",
    "Review_Length",
    "Title_Length"
]

# Separate input features and target
X = df[selected_features].copy()

# Encode sentiment labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["Sentiment"])

print("FEATURES AND LABELS PREPARED")
print("=" * 50)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nLabel encoding:")
for index, label in enumerate(label_encoder.classes_):
    print(f"{label}: {index}")

FEATURES AND LABELS PREPARED
Feature matrix shape: (16155, 8)
Target shape: (16155,)

Label encoding:
NEGATIVE: 0
NEUTRAL: 1
POSITIVE: 2


In [4]:
from sklearn.model_selection import StratifiedGroupKFold

# Configure the same grouped split as the original experiment
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Generate the first 80/20 split
train_indices, test_indices = next(
    splitter.split(X, y, groups=X["Text"])
)

# Create training and testing datasets
X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_train = y[train_indices]
y_test = y[test_indices]

# Check for duplicate review text leakage
overlap = set(X_train["Text"]).intersection(set(X_test["Text"]))

print("HYBRID EXPERIMENT TRAIN-TEST SPLIT")
print("=" * 55)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Identical review texts across sets:", len(overlap))

print("\nTraining sentiment distribution:")
print(pd.Series(label_encoder.inverse_transform(y_train)).value_counts())

print("\nTesting sentiment distribution:")
print(pd.Series(label_encoder.inverse_transform(y_test)).value_counts())

HYBRID EXPERIMENT TRAIN-TEST SPLIT
Training records: 12924
Testing records: 3231
Identical review texts across sets: 0

Training sentiment distribution:
POSITIVE    10432
NEGATIVE     1282
NEUTRAL      1210
Name: count, dtype: int64

Testing sentiment distribution:
POSITIVE    2608
NEGATIVE     320
NEUTRAL      303
Name: count, dtype: int64


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# BRANCH 1: TEXT-BASED LOGISTIC REGRESSION
# ==========================================

text_preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(
                max_features=3000,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            ),
            "Text"
        ),
        (
            "title",
            TfidfVectorizer(
                max_features=1000,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            ),
            "Title"
        )
    ]
)

text_model = Pipeline([
    ("preprocessor", text_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])


# ==========================================
# BRANCH 2: METADATA-BASED RANDOM FOREST
# ==========================================

categorical_features = [
    "Location_Type",
    "Province",
    "Location_Name",
    "User_Country"
]

numerical_features = [
    "Review_Length",
    "Title_Length"
]

metadata_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

metadata_model = Pipeline([
    ("preprocessor", metadata_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ))
])

print("BOTH HYBRID BRANCHES CONFIGURED SUCCESSFULLY")
print("Text branch: TF-IDF + Logistic Regression")
print("Metadata branch: One-Hot Encoding + Random Forest")

BOTH HYBRID BRANCHES CONFIGURED SUCCESSFULLY
Text branch: TF-IDF + Logistic Regression
Metadata branch: One-Hot Encoding + Random Forest


In [6]:
import time

# ==========================================
# TRAIN TEXT BRANCH
# ==========================================

print("TRAINING TEXT BRANCH...")
start = time.time()

text_model.fit(X_train, y_train)

text_training_time = time.time() - start

print(f"Text branch training time: {text_training_time:.2f} seconds")


# ==========================================
# TRAIN METADATA BRANCH
# ==========================================

print("\nTRAINING METADATA BRANCH...")
start = time.time()

metadata_model.fit(X_train, y_train)

metadata_training_time = time.time() - start

print(f"Metadata branch training time: {metadata_training_time:.2f} seconds")


print("\nBOTH HYBRID BRANCHES TRAINED SUCCESSFULLY")

TRAINING TEXT BRANCH...
Text branch training time: 2.15 seconds

TRAINING METADATA BRANCH...
Metadata branch training time: 19.13 seconds

BOTH HYBRID BRANCHES TRAINED SUCCESSFULLY


In [7]:
# Generate probability predictions from both branches

text_probabilities = text_model.predict_proba(X_test)

metadata_probabilities = metadata_model.predict_proba(X_test)

print("TEXT BRANCH PROBABILITY SHAPE:", text_probabilities.shape)
print("METADATA BRANCH PROBABILITY SHAPE:", metadata_probabilities.shape)

print("\nCLASS ORDER:")
print("Text branch:", text_model.classes_)
print("Metadata branch:", metadata_model.classes_)

print("\nPROBABILITY PREDICTIONS GENERATED SUCCESSFULLY")

TEXT BRANCH PROBABILITY SHAPE: (3231, 3)
METADATA BRANCH PROBABILITY SHAPE: (3231, 3)

CLASS ORDER:
Text branch: [0 1 2]
Metadata branch: [0 1 2]

PROBABILITY PREDICTIONS GENERATED SUCCESSFULLY


In [8]:
# ==========================================
# HYBRID ENSEMBLE: SOFT VOTING
# ==========================================

# Combine probabilities from both branches
hybrid_probabilities = (
    text_probabilities + metadata_probabilities
) / 2

# Select the sentiment with the highest combined probability
hybrid_predictions = np.argmax(
    hybrid_probabilities,
    axis=1
)

print("HYBRID ENSEMBLE CREATED SUCCESSFULLY")
print("Number of predictions:", len(hybrid_predictions))

print("\nHYBRID PREDICTION DISTRIBUTION:")
print(
    pd.Series(
        label_encoder.inverse_transform(hybrid_predictions)
    ).value_counts()
)

print("\nHYBRID PROBABILITY SHAPE:", hybrid_probabilities.shape)

HYBRID ENSEMBLE CREATED SUCCESSFULLY
Number of predictions: 3231

HYBRID PREDICTION DISTRIBUTION:
POSITIVE    2569
NEGATIVE     345
NEUTRAL      317
Name: count, dtype: int64

HYBRID PROBABILITY SHAPE: (3231, 3)


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ==========================================
# HYBRID MODEL EVALUATION
# ==========================================

hybrid_accuracy = accuracy_score(y_test, hybrid_predictions)

hybrid_precision = precision_score(
    y_test, hybrid_predictions, average="macro"
)

hybrid_recall = recall_score(
    y_test, hybrid_predictions, average="macro"
)

hybrid_macro_f1 = f1_score(
    y_test, hybrid_predictions, average="macro"
)

hybrid_weighted_f1 = f1_score(
    y_test, hybrid_predictions, average="weighted"
)

print("=" * 55)
print("HYBRID ENSEMBLE RESULTS")
print("=" * 55)

print(f"Accuracy:          {hybrid_accuracy:.4f}")
print(f"Precision (Macro): {hybrid_precision:.4f}")
print(f"Recall (Macro):    {hybrid_recall:.4f}")
print(f"F1 Score (Macro):  {hybrid_macro_f1:.4f}")
print(f"F1 Score (Weighted): {hybrid_weighted_f1:.4f}")

print("\nDETAILED CLASSIFICATION REPORT")
print("=" * 55)

print(
    classification_report(
        y_test,
        hybrid_predictions,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

print("\nCONFUSION MATRIX")
print("=" * 55)

print(confusion_matrix(y_test, hybrid_predictions))

HYBRID ENSEMBLE RESULTS
Accuracy:          0.8066
Precision (Macro): 0.5841
Recall (Macro):    0.5984
F1 Score (Macro):  0.5909
F1 Score (Weighted): 0.8091

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

    NEGATIVE       0.54      0.58      0.56       320
     NEUTRAL       0.31      0.33      0.32       303
    POSITIVE       0.90      0.89      0.90      2608

    accuracy                           0.81      3231
   macro avg       0.58      0.60      0.59      3231
weighted avg       0.81      0.81      0.81      3231


CONFUSION MATRIX
[[ 185   47   88]
 [  45   99  159]
 [ 115  171 2322]]


In [10]:
# Load the original seven-model comparison results

results_path = "../results/model_comparison.csv"

original_results = pd.read_csv(results_path)

print("ORIGINAL MODEL COMPARISON")
print("=" * 60)

display(original_results)

print("\nAVAILABLE COLUMNS:")
print(original_results.columns.tolist())

ORIGINAL MODEL COMPARISON


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time (s),Prediction Time (s)
0,Decision Tree,0.558651,0.452148,0.527666,0.428451,0.625850,1.651871,0.002409
1,Bagging,0.724853,0.554679,0.598729,0.553173,0.755936,13.275867,0.072185
2,Random Forest,0.820489,0.619188,0.642574,0.629875,0.825363,1.239802,0.040318
3,AdaBoost,0.812442,0.570632,0.351956,0.333971,0.733745,8.281451,0.065065
4,Gradient Boosting,0.829774,0.587072,0.422227,0.438533,0.772093,23.611833,0.012262
5,XGBoost,0.775302,0.583854,0.669824,0.610141,0.799212,75.847050,0.040961
6,Stacking,0.740638,0.594102,0.665971,0.600770,0.780964,37.937398,0.394087



AVAILABLE COLUMNS:
['Model', 'Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Weighted F1', 'Training Time (s)', 'Prediction Time (s)']


In [11]:
# ==========================================
# COMBINE ORIGINAL AND HYBRID RESULTS
# ==========================================

hybrid_result = pd.DataFrame([{
    "Model": "Hybrid (Text + Metadata)",
    "Accuracy": hybrid_accuracy,
    "Macro Precision": hybrid_precision,
    "Macro Recall": hybrid_recall,
    "Macro F1": hybrid_macro_f1,
    "Weighted F1": hybrid_weighted_f1,
    "Training Time (s)": (
        text_training_time + metadata_training_time
    )
}])

# Combine with the original seven models
combined_results = pd.concat(
    [original_results, hybrid_result],
    ignore_index=True
)

# Sort by Macro F1
combined_results = combined_results.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

print("=" * 75)
print("ENSEMBLELAB - EIGHT-MODEL PERFORMANCE COMPARISON")
print("=" * 75)

display(
    combined_results.round(4)
)

ENSEMBLELAB - EIGHT-MODEL PERFORMANCE COMPARISON


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time (s),Prediction Time (s)
0,Random Forest,0.8205,0.6192,0.6426,0.6299,0.8254,1.2398,0.0403
1,XGBoost,0.7753,0.5839,0.6698,0.6101,0.7992,75.8470,0.0410
2,Stacking,0.7406,0.5941,0.6660,0.6008,0.7810,37.9374,0.3941
3,Hybrid (Text + Metadata),0.8066,0.5841,0.5984,0.5909,0.8091,21.2858,NaN
4,Bagging,0.7249,0.5547,0.5987,0.5532,0.7559,13.2759,0.0722
5,Gradient Boosting,0.8298,0.5871,0.4222,0.4385,0.7721,23.6118,0.0123
6,Decision Tree,0.5587,0.4521,0.5277,0.4285,0.6258,1.6519,0.0024
7,AdaBoost,0.8124,0.5706,0.3520,0.3340,0.7337,8.2815,0.0651


In [12]:
import time

# Measure prediction time for both branches and fusion
start = time.perf_counter()

text_probabilities = text_model.predict_proba(X_test)
metadata_probabilities = metadata_model.predict_proba(X_test)

hybrid_probabilities = (
    text_probabilities + metadata_probabilities
) / 2

hybrid_predictions = np.argmax(hybrid_probabilities, axis=1)

hybrid_prediction_time = time.perf_counter() - start

print("HYBRID PREDICTION TIME")
print("=" * 45)
print(f"Total prediction time: {hybrid_prediction_time:.4f} seconds")
print(f"Average time per review: {hybrid_prediction_time / len(X_test):.6f} seconds")
print("Number of test reviews:", len(X_test))

HYBRID PREDICTION TIME
Total prediction time: 0.4509 seconds
Average time per review: 0.000140 seconds
Number of test reviews: 3231


In [18]:
import os

# Create results directory if necessary
os.makedirs("../results", exist_ok=True)

# Update hybrid result with prediction time
hybrid_result["Prediction Time (s)"] = hybrid_prediction_time

# Rebuild the complete comparison
combined_results = pd.concat(
    [original_results, hybrid_result],
    ignore_index=True
)

combined_results = combined_results.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

# Save separate hybrid comparison file
combined_results.to_csv(
    "../results/hybrid_model_comparison.csv",
    index=False
)

# Save hybrid model evaluation details
hybrid_evaluation = {
    "accuracy": hybrid_accuracy,
    "macro_precision": hybrid_precision,
    "macro_recall": hybrid_recall,
    "macro_f1": hybrid_macro_f1,
    "weighted_f1": hybrid_weighted_f1,
    "training_time": text_training_time + metadata_training_time,
    "prediction_time": hybrid_prediction_time,
    "confusion_matrix": confusion_matrix(
        y_test, hybrid_predictions
    ),
    "classification_report": classification_report(
        y_test,
        hybrid_predictions,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )
}

joblib.dump(
    hybrid_evaluation,
    "../results/hybrid_evaluation.pkl"
)

print("HYBRID RESULTS SAVED SUCCESSFULLY")
print("\nFiles created:")
print("../results/hybrid_model_comparison.csv")
print("../results/hybrid_evaluation.pkl")

print("\nFINAL MODEL RANKING BY MACRO F1:")
display(combined_results.round(4))

HYBRID RESULTS SAVED SUCCESSFULLY

Files created:
../results/hybrid_model_comparison.csv
../results/hybrid_evaluation.pkl

FINAL MODEL RANKING BY MACRO F1:


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time (s),Prediction Time (s)
0,Random Forest,0.8205,0.6192,0.6426,0.6299,0.8254,1.2398,0.0403
1,XGBoost,0.7753,0.5839,0.6698,0.6101,0.7992,75.8470,0.0410
2,Stacking,0.7406,0.5941,0.6660,0.6008,0.7810,37.9374,0.3941
3,Hybrid (Text + Metadata),0.8066,0.5841,0.5984,0.5909,0.8091,21.2858,0.4509
4,Bagging,0.7249,0.5547,0.5987,0.5532,0.7559,13.2759,0.0722
5,Gradient Boosting,0.8298,0.5871,0.4222,0.4385,0.7721,23.6118,0.0123
6,Decision Tree,0.5587,0.4521,0.5277,0.4285,0.6258,1.6519,0.0024
7,AdaBoost,0.8124,0.5706,0.3520,0.3340,0.7337,8.2815,0.0651


In [17]:
import joblib

joblib.dump(
    hybrid_evaluation,
    "../results/hybrid_evaluation.pkl"
)

print("HYBRID EVALUATION SAVED SUCCESSFULLY")

HYBRID EVALUATION SAVED SUCCESSFULLY


In [19]:
import joblib
import os

# Ensure the models directory exists
os.makedirs("../models", exist_ok=True)

# Save the trained text-based model
joblib.dump(
    text_model,
    "../models/hybrid_text_model.pkl"
)

# Save the trained metadata-based model
joblib.dump(
    metadata_model,
    "../models/hybrid_metadata_model.pkl"
)

print("HYBRID MODELS SAVED SUCCESSFULLY")

print("\nSaved files:")
print("../models/hybrid_text_model.pkl")
print("../models/hybrid_metadata_model.pkl")

HYBRID MODELS SAVED SUCCESSFULLY

Saved files:
../models/hybrid_text_model.pkl
../models/hybrid_metadata_model.pkl


In [20]:
import joblib
import os

text_model_path = "../models/hybrid_text_model.pkl"
metadata_model_path = "../models/hybrid_metadata_model.pkl"

print("TEXT MODEL EXISTS:", os.path.exists(text_model_path))
print("METADATA MODEL EXISTS:", os.path.exists(metadata_model_path))

# Load both saved models
loaded_text_model = joblib.load(text_model_path)
loaded_metadata_model = joblib.load(metadata_model_path)

print("\nBOTH HYBRID MODELS LOADED SUCCESSFULLY")
print("Text model:", type(loaded_text_model).__name__)
print("Metadata model:", type(loaded_metadata_model).__name__)

TEXT MODEL EXISTS: True
METADATA MODEL EXISTS: True

BOTH HYBRID MODELS LOADED SUCCESSFULLY
Text model: Pipeline
Metadata model: Pipeline
